# GridMind GR — Load Forecast Model

Evaluation and training of the final v1 Random Forest model for system load forecasting.

The notebook uses the same six-feature contract as the production forecasting module.


In [ ]:
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

sys.path.append("../src")

from gridmind.models.load_forecast import FEATURE_COLS, load_model_data


In [ ]:
df = load_model_data()

train = df[df["timestamp"] < "2026-01-01"].copy()

validation = df[
    (df["timestamp"] >= "2026-01-01")
    & (df["timestamp"] < "2026-07-01")
].copy()

test = df[df["timestamp"] >= "2026-07-01"].copy()

print(f"Train:      {len(train):,} rows")
print(f"Validation: {len(validation):,} rows")
print(f"Test:       {len(test):,} rows")


In [ ]:
def evaluate(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    wape = np.abs(y_true - y_pred).sum() / np.abs(y_true).sum() * 100

    return {
        "MAE": round(mae, 2),
        "RMSE": round(rmse, 2),
        "WAPE": round(wape, 2),
    }


## Baselines


In [ ]:
baseline_24 = evaluate(
    validation["target"],
    validation["load_lag_24"],
)

baseline_168 = evaluate(
    validation["target"],
    validation["load_lag_168"],
)

baseline_results = pd.DataFrame(
    {
        "24h Baseline": baseline_24,
        "168h Baseline": baseline_168,
    }
).T

baseline_results


## Random Forest


In [ ]:
RF_PARAMS = {
    "n_estimators": 300,
    "max_depth": 20,
    "min_samples_leaf": 2,
    "random_state": 42,
    "n_jobs": -1,
}

model = RandomForestRegressor(**RF_PARAMS)
model.fit(train[FEATURE_COLS], train["target"])

validation_prediction = model.predict(validation[FEATURE_COLS])
validation_metrics = evaluate(
    validation["target"],
    validation_prediction,
)

pd.DataFrame(
    {
        "24h Baseline": baseline_24,
        "Random Forest": validation_metrics,
    }
).T.sort_values("WAPE")


In [ ]:
sample = validation[
    (validation["timestamp"] >= "2026-02-01")
    & (validation["timestamp"] < "2026-02-08")
].copy()

sample["prediction"] = model.predict(sample[FEATURE_COLS])

plt.figure(figsize=(14, 5))
plt.plot(sample["timestamp"], sample["target"], label="Actual")
plt.plot(sample["timestamp"], sample["prediction"], label="Random Forest")
plt.title("Actual vs Random Forest Load — Sample Week")
plt.xlabel("Timestamp")
plt.ylabel("Load (MWh)")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## Final test

After selecting the model on the validation period, it is refit on train + validation and evaluated once on the held-out July–August 2026 test period.


In [ ]:
train_validation = pd.concat(
    [train, validation],
    ignore_index=True,
)

test_model = RandomForestRegressor(**RF_PARAMS)
test_model.fit(
    train_validation[FEATURE_COLS],
    train_validation["target"],
)

test_prediction = test_model.predict(test[FEATURE_COLS])
test_metrics = evaluate(test["target"], test_prediction)

final_results = pd.DataFrame(
    {
        "24h Baseline - Validation": baseline_24,
        "Random Forest - Validation": validation_metrics,
        "Random Forest - Test": test_metrics,
    }
).T

final_results


## Production model

The evaluation above remains untouched. The production artifact is then trained on all currently available model-ready history so future forecasts use the latest actual data.


In [ ]:
production_model = RandomForestRegressor(**RF_PARAMS)
production_model.fit(
    df[FEATURE_COLS],
    df["target"],
)

model_path = Path("../models/random_forest_load_forecast.joblib")
model_path.parent.mkdir(parents=True, exist_ok=True)

joblib.dump(
    {
        "model": production_model,
        "features": FEATURE_COLS,
    },
    model_path,
)

print(f"Saved production model: {model_path}")
print(f"Training rows: {len(df):,}")
print(f"Features: {FEATURE_COLS}")


### Model conclusion

The final GridMind v1 load forecaster is a Random Forest using six day-ahead-safe features. The 24-hour persistence baseline remains the primary benchmark, while monthly walk-forward performance is evaluated separately in `load_backtest.py`.
